# Item 99: Consider `memoryview` and `bytearray` for Zero-Copy

Interactions with `bytes`

## Notes

-   Python requires extra effort to parallelise CPU-bound computation
    (See [Item 79](../../Chapter_09/Item_079/item_079.qmd) and [Item
    94](../Item_094/item_094.qmd))
-   But, can support high-throughput parallel I/O (See [Item
    68](../../Chapter_09/Item_068/item_068.qmd) and [Item
    75](../../Chapter_09/Item_075/item_075.qmd))
-   However, understanding the tools available and how to use them
    *without* leading to slow code can require some skill
-   For example, consider a media-streaming server
    -   Users don’t need to download a video in advance
    -   Users can move forward or backward within a video
-   We might have functions to implement this by converting a time-code
    to a index and returning the associated chunk of data

In [1]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    # simulate by returning random data
    return os.urandom(size)


video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode)
size = (8**2)
video_data = request_chunk(video_id, byte_offset, size)

print(f"{video_id=}, {timecode=}, {byte_offset=}, {video_data=}")

video_id=1, timecode='01:09:14:28', byte_offset=0, video_data=b'\x97\xe9\x83%\xd8\x81\xb22\x99\xc5M\x13\x98\xe5\xcd\xe9\x0f\x1e\x94\xda\xb8I>\xe3\x0f\xf9?\xb2\xc6\xa1J\x9cD\x0f\xc5\x7f\xa4\xa8\xcf\xab)\xe8y\x7f\xa6\x1eV\xb2\xb7\xf9\x1d\r\xca\x7f\xd7\xf32:\x8e\x95rMuI'

-   How do we now implement the server-side handler that receives
    `request_chunk`
    -   Must then return the associated video data chunk
-   First we assume that the program is driven by an `asyncio` process
    (See [Item 76](../../Chapter_09/Item_076/item_076.qmd))
    -   Now want to focus on how to handle extracting the chunk
    -   Assume video is cached memory
    -   Extracted then sent over a socket back to a client

In [2]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    return video_data[byte_offset : byte_offset + size]

# Adding in the handling

# simulate a socket connection
class NullSocket:
    def __init__(self):
        self.handle = open(os.devnull, "wb")

    def send(self, data):
        self.handle.write(data)

socket = NullSocket() # represents client socket connection
size = (8 ** 2) # Requested chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video_id

video_id = 1
timecode = "01:09:14:28"

byte_offset = timecode_to_index(video_id, timecode)
chunk = request_chunk(video_id, byte_offset, size)
socket.send(chunk)

print(f"Sent {chunk=} over socket")

Sent chunk=b'\xf4\xd9\x9b\xc5\xe9\xd8O\x1cR\xfa\xbfYc\xb0\xe1\xaf\xe3\x00)K8\x8f\xbcsV\xb6V\xc6\xfa\xcaS\xdfj\xdf!\xb1:h3\xfc\xac~@\xea\x9a\xcd\x06\xd4\x94\xd7\xcd\xdc\xf0\x0b\x98\x8e\xa6\xd3\xbdu?\xd9\x13B' over socket

-   Latency and throughput determined by two factors
    1.  How long to slice (See [Item
        14](../../Chapter_02/Item_014/item_014.qmd)) the chunk from
        `video_data`
    2.  How long to transmit over a socket
-   Focusing just on point 1, we can microbenchmark how long fetching a
    chunk takes.
    -   We’ll also exclude the function call wrapper
    -   Here we’ll set the size to $20$ MB.

In [3]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
byte_offset = 0

def run_test():
    chunk = video_data[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000882980 seconds

-   This takes about $5$ milliseconds
-   Theoretical server maximum throughput is thus, limited by video
    extraction speed as

$$
\begin{align}
    \frac{20 \text{ MB}}{5 \text{ ms}} &= 4 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Server also limited to,

$$
\begin{align}
    \frac{1 \text{ CPU=second}}{5 \text{ ms}} &= 200 \text{ clients in parallel}
\end{align}
$$

-   But we already know that `asyncio` should be able to scale up to
    tens of thousands of simultaneous connections
-   The slowdown is because as discussed slices create copies
    -   Copying consumes CPU time
-   Instead we can use `memoryview`
    -   A built-in type for handling the CPython `buffer` protocol
        -   Low-level C API allowing Python runtime and C extensions
            (See [Item 96](../Item_096/item_096.qmd)) to access
            underlying data buffers
            -   Can then treat them as `bytes` instances
        -   Since Python 3.12 the buffer protocol is also emulatable in
            python
-   `memoryview` can be sliced to create a new `memoryview` without a
    copy

In [4]:
data = b"shave and a haircut, two bits"
view = memoryview(data)
chunk = view[12:19]

print(chunk)
print("Size:            ", chunk.nbytes)
print("Data in view:    ", chunk.tobytes())
print("Underlying data: ", chunk.obj)

Size:             7
Data in view:     b'haircut'
Underlying data:  b'shave and a haircut, two bits'

-   These *zero-copy* operations can significantly speed-up code that
    heavily processes memory, e.g.
    1.  I/O-bound access
    2.  Heavy numerical mathematics (e.g. Numpy)
-   Using `memoryview` as a drop-in replacement for our video serving
    service

In [5]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
video_view = memoryview(video_data)
byte_offset = 0

def run_test():
    chunk = video_view[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000000204 seconds

-   This should run in a several hundred nanoseconds
-   So an order of magnitude faster than the `bytes` slicing technique
-   Our new theoretical maximum throughput is then

$$
\begin{align}
\frac{20 \text{ MB}}{250 \text{ ns}} &= 80 \text{ TB}\text{s}^{-1}
\end{align}
$$

-   Or in terms of parallel clients

$$
\begin{align}
\frac{1 \text{ CPU-second}}{250 \text{ ns}} &= 4 \times 10^{9}
\end{align}
$$

-   So four million clients. Now the program should be bound by the
    socket performance rather than CPU constraints.

-   Now consider a reversed process

    -   Users must submit live video streams that are then broadcast out
        to viewers

-   We need to store incoming video data

    -   Cache it for clients to read from

In [6]:
import os

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder

# socket connection from client


size = (4 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]

video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode) # Incoming buffer position
video_view = memoryview(video_cache)


class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
chunk = socket.recv(size)
before = video_view[:byte_offset]
after = video_view[byte_offset + size:]

new_cache = b"".join([before, chunk, after])

print(f"Updated the cache: {new_cache=}")

Updated the cache: new_cache=b'\x82\x99\x11i1\x87Y\x9d\xebP{\xbc=,3\xea!W\xb6\xf8\xdd\x00p\xeb\xbc\xdf\x0c\xde\x18\xd0\x1f\xe2\x0e\x98\xb3\x0f\x1f\xab\x90h\xff\x87S}\xd2\x12\x80\xc3C|M<\xd6z\xb5\xf6Xad\x9f\x9b\xc1Fk\xdbK3\xfd\x08\x87}\x9b\xb6ki?\x06\xa45\x829\xd1O\xaeS~Qz\xbfX\x85\x93\xdb\x19a\xdf]\xb2\x8a.\x0cx\x98\xd2\x13\xb2\x13\xfb\xca9{\xe7\x1e`H\x13(\x91d\x80\xde7?\x14\x94\xb2,\x8d\xde\xd4\xf7r\xa8g\xd2\xbb\xd0\x1b\xc0\xa5\x98\xf7\xa6\x84\xe4)s\x82\xc6\xe7\xb1\x95:\xfe\xdb*\x8a!\xe5\xe3D+\xcfk\xa0\xbe)3\xa3\xdc\xf7H\x80\x8eN\x992\x8a\xcd;\x80\xf3\x04aM\xa5\xb3\xfa\\u\xe2+\x12j\xcc\xdd\xc5f\xec\x1fh\x12\x80\xa5L\xda\x83\x95\x9c\xe17ALo\xbfN\n\xaeK\x92\x1e)y\xf6\xf6\xce\x9a\x9b\x9b\x1f\x8f\x8c5Q\x1e]?\nf5\xeb\x9aO\xf2(\x18u=\x94\xdf@\x91\xd1X\xbf\x9b\x040\x9e\x02\x15p\x9e\\\x9f\xaf\xa5\x7f\xbb\xccBAB \x16 \xc2\xeac7%:\tV\xf1v\x84\x7f\x9e\xcfI\xee0\xc2\xc4d\xc1^\x97\xd7\x9c\x1d\xe4\x16+M\xc7\xce\xd6g\x06\x85\xf8\xb6$X\xbaR\xa9z'

-   `socket.recv` returns a `bytes` instance
    -   Splice this into the existing cache
    -   Insert at the current `byte_offset` via slicing and `bytes.join`
-   Now need to profile the timing

In [7]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
size = (1024 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)
byte_offset = 1234 # pick arbitrary point in the middle

def run_test():
    chunk = socket.recv(size)
    before = video_view[:byte_offset]
    after = video_view[byte_offset + size : ]
    new_cache = b"".join([before, chunk, after])

result = (timeit.timeit(stmt="run_test()", globals=globals(), number=100,) / 100)

print(f"{result:0.9f} seconds")

0.000915514 seconds

-   This takes about three milliseconds to receive $1$ MB and update the
    cache.
-   Maximum throughput to receive is then

$$
\begin{align}
\frac{1 \text{ MB}}{ 3 \text{ ms}} &\approx 330 \text{ MB}\text{s}^{-1}
\end{align}
$$

-   Means we are limited to about $300$ simultaneously streaming clients
-   Can use `bytearray` instead of `memoryview`
    -   `bytes` are immutable like strings

In [8]:
some_bytes = b"hello"
some_bytes[0] = 0x79

-   `bytearray` is effectively a mutable version of `bytes`
    -   Can overwrite indices
-   `bytearray` values are integers rather than bytes

In [9]:
array = bytearray(b"hello")
array[0] = 0x79
print(array)

bytearray(b'yello')

-   Can still wrap a `bytearray` in a `memoryview` to avoid extra copies
    -   Then can slice the `memoryview` and modify to overwrite the
        underlying `bytearray`

In [10]:
array = bytearray(b"row, row, row your boat")
view = memoryview(array)
write_view = view[3:13]
write_view[:] = b"-10 bytes-"
print(array)

bytearray(b'row-10 bytes- your boat')

-   Library methods in Python user the buffer protocol for fast data
    receipt or reading, e.g.
    1.  `socket.recv_into`
    2.  `RawIOBase.read_into`
-   These methods avoid creating copies and allocating memory
    -   Received data goes into existing buffer
-   We can convert our program to use `recv_into` and a `memoryview`
    slice to speed up our broadcasting method

In [11]:
class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (4 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)

chunk = write_view[byte_offset : byte_offset + size]
socket.recv_into(chunk)

print(f"New cache: {video_cache=}")

New cache: video_cache=b'\xc2\xe2\xf7\x95\x8a\xf3FE4S\xd9f@\xe9|\x99C,\x02\x88\xa5\xa9\xbf\x04\x06Q#\xdb\x14\xc9\xfe#\xf3\xfa\x04(\xe4g:\t\x8a\xf5\x84\xc2~v\xbel\x04tC\x0e\xab\x15\xec\xa4\x1a\xa2O\xa3z\xb7r\x98\x04l1\xf9l^\xdf\xfdH\xe8!f\xf2\xa5\xb6\x91Q\xf6\x017^\xcb\xd2\xa0i\xd2 3\xf3hK\x02\x9c\xad\xa3V\x8ft\xb2\'\xf7\x8fFE>\x9euqj\xc4\xbc<\xe9\xdf\xe6\x84\xf3\xca\x0c\x14Nk\xb8&d\xe9\x1bD\x0e\x8b-,\xfez\xd5\x9d\x81\xeb\xd8\x8e\xbex\x02\xc7\x1d\xc9b\xc9\xa5\xc2\xf0\x07_^\x166\xd5\x03 \x02g9F|%i\x10\x02pZ\xfd\xd7\x8b\'V\xca\xe28\xaa\xe2\xca\x01t\x07w\xb4\xa3\xdd~\x11\x8d\xd5H\x1aZ\t\xd8\xcd#\xd03Z\xcbl\x1b\xc0\x89\xad\xe2$\xa7\x8c\x1a\x9d\x93\x8dbg7\x91\xd6:J\xbc\xf4\xd1\xd3\x8d-^\x8b\n!\x17\'\x0e\xdb\x16R\x8b\xe92\xd9\x84\x13L\xa8\x07N\x1e\x8d\x91,\xfa/\xb4\xa7P3\xc38\xf953\xc3\xb7\xae\xfc\xec\x19\xff\x87.\xf9C\xccgI\xcd\x16\xd7M\xdax_z\x07L#\xae\x064A\x01"\xf5\x90\xad\x07\x92\xa2\xa5<\xb3\xce\x16)\xb2w\x843\xc2CY\xbe,'

-   We can again microbenchmark the result for a $1$ MB chunk

In [12]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (1024 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)


def run_test():
    chunk = write_view[byte_offset : byte_offset + size]
    socket.recv_into(chunk)

result = (
    timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100
)

print(f"{result:0.9f} seconds")

0.000040038 seconds

-   On my machine this takes about $90 \;\mu\text{s}$. Which means we
    could support,

$$
\begin{align}
    \frac{1 \text{ MB}}{90 \; \mu\text{s}} &= 11 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Which also supports,

$$
\begin{align}
    \frac{11 \text{ GB}}{1 \text{MB}} &= 11,000 \text{ processes}
\end{align}
$$

-   Much better scalability

## Things to Remember

-   `memoryview` provides zero-copy methods for reading and writing to
    slices of objects supporting the buffer protocol
-   `bytearray` built-in provides a mutable `bytes`-like type
    -   Can be used for zero-copy data reads
    -   Works with functions like `socket.recv_into`
-   `memoryview` can wrap a `bytearray`
    -   Let’s received data to be spliced into an existing buffer
    -   No need for extra copies